<a href="https://colab.research.google.com/github/Githaigaa/SemProject_MLEngine/blob/main/AfriBERTa_Base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Misinformation Detection**

**Cell 1: Load and Inspect the raw dataset**

Before any modelling we need to know the size of the data, whether anthing is missing, and whether the classes are balanced.

In [1]:
import pandas as pd

#Dataset file
try:
    df = pd.read_csv("Dataset(Train+Test).csv", encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv("Dataset(Train+Test).csv", encoding='latin-1')

#Basic info
print("Shape:", df.shape)
print("\ncolumns:", df.columns.tolist())
print("\nData type:\n", df.dtypes)
print("\nFirst 5 rows:\n", df.head())
print("\nMissing values:\n", df.isnull().sum())
print("\nClass distriution:")
for col in df.columns:
  if df[col].nunique() < 10:
    print(f"\n{col}:\n", df[col].value_counts())

Shape: (9000, 6)

columns: ['Unnamed: 0', 'tweet', 'label', 'text', 'language', 'contains_emoji']

Data type:
 Unnamed: 0         int64
tweet             object
label             object
text              object
language          object
contains_emoji      bool
dtype: object

First 5 rows:
    Unnamed: 0                                              tweet    label  \
0        5266  @user @user So he only worked from June 1st to...     fact   
1        4574  It s 61 days to 2022 general elections to end ...     fact   
2          96        @user Kenya Kwanza yaongezewa @user za tena  neutral   
3        1967  There will be no need of panic\nAvoid incompet...     fact   
4         118  IEBC Kenya imeweka mbinu mbili kuu za kuwatamb...     fact   

                                                text language  contains_emoji  
0  worked june st alter presidential elections re...    sheng            True  
1  s days general elections end martharao sababu ...    sheng           False  
2     

**Cell 2: Clean the text and remove duplicates**

Step 1: Drop the unamed: 0 column
Step 2:clean_tweet() builds a new column tweet_clean from the raw tweet. It does the following:
1. Empty values become an empty string.
2. Lowercases everything
3. Replaces links with the token.
4. Replaces *@mentions* with the token *user*
5. Turns hashtags into plain words.
6. Replaces emojis with the token *emoji*.
7. Collapses newlines and repeated spaces.
8. Removes remaining punctuation but keeps apostrophes.

Step 3: Remove exact duplicate on the pair(tweet_clean, label). This prevents data leakage whereby the same tweet appears in both training and test data.
Step 4: Remove very short rows. They carry too little meaning to classify.
Step 5: Re-check the label and language counts after cleaning.

In [2]:
import pandas as pd
import re
import numpy as np

#Load the dataset
#df = pd.read_csv("final_Dataset.csv")

#1. Drop the reduntant columns
df = df.drop(columns =['Unnamed: 0'])

#2. Reclean the raw 'tweet' column
def clean_tweet(text):
  if pd.isna(text):
    return ""

  #Lowercase
  text = text.lower()

  #Replace the URLs with a token
  text = re.sub(r'https?://\S+', ' url ', text)

  #Replace @mentions with a token
  text = re.sub(r'@\w+', ' user ', text)

  #Split CamelCase and keep the word
  def split_hashtag(match):
    tag = match.group(1)
    #insert space before capitals and between letters/digits
    tag = re.sub(r'([a-z])([A-Z])', r'\1 \2', tag)
    tag = re.sub(r'([A-zA-Z])(\d)', r'\1 \2', tag)
    return ' ' + tag.lower() + ' '
  text = re.sub(r'#(\w+)', split_hashtag, text)
  #Replace emojis with a token (Keeps signal without breaking tokenizer)
  text = re.sub(r'[\U0001F300-\U0001FAFF\U00002700-\U000027BF]', ' emoji ', text)
  #Replace newlines and multiple whitespace with space
  text = re.sub(r'\s+', ' ', text)
  #Remove lefover punctuation but keep apostrophes for contractions
  text = re.sub(r'[^\w\s\']', ' ', text)
  #Collapse again after punctuation removal
  text = re.sub(r'\s+', ' ', text).strip()
  return text

df['tweet_clean'] = df['tweet'].apply(clean_tweet)

#Sanity check: view a few examples
print("===Before===")
print(df['tweet'].iloc[3])
print("\n===After (tweet_clean) ===")
print(df['tweet_clean'].iloc[3])
print("\n=== Existing 'text' column ===")
print(df['text'].iloc[3])

#3. Remove exact duplicates (on the cleaned text + labels)
before = len(df)
df = df.drop_duplicates(subset=['tweet_clean', 'label'], keep='first').reset_index(drop=True)
after = len(df)
print(f"\n[De-dup] Removed {before - after} duplicates. Remaining: {after}")
#print(f"[De-dup] SKIPPED to preserve language balance. Keeping all {before} rows.")

#4. Removing otliers: drop empty or near-empty text
df = df[df['tweet_clean'].str.split().str.len() >= 3].reset_index(drop=True)
print(f"[Outlier removal] Remaining after dropping <3-word rows: {len(df)}")

#5. Check the class/language distribution AFTER cleaning.
print("\n[After cleaning] labels distribution:")
print(df['label'].value_counts())
print("\n[After cleaning] Language distribution:")
print(df['language'].value_counts())



===Before===
There will be no need of panic
Avoid incompetence and vote wisely
KenyaDecides2022 AmaniNiJukumuLangu https://t.co/G8PnWJYN91

===After (tweet_clean) ===
there will be no need of panic avoid incompetence and vote wisely kenyadecides2022 amaninijukumulangu url

=== Existing 'text' column ===
need panicavoid incompetence vote wiselykenyadecides2022 amaninijukumulangu

[De-dup] Removed 2761 duplicates. Remaining: 6239
[Outlier removal] Remaining after dropping <3-word rows: 6236

[After cleaning] labels distribution:
label
neutral    2189
fake       2024
fact       2023
Name: count, dtype: int64

[After cleaning] Language distribution:
language
sheng      2999
English    2999
Swahili     238
Name: count, dtype: int64


From the Output:

2761 duplicates were removed(which are essentially all Swahili) and 3 very short rows were dropped.
De-duplication was necessary, because keeping the copies would let the model memorise repeats and leak them across the train/test split.
The remaining 238 unique tweets are insufficient to to train the model.

**Cell 3: Encode labels and split the data**



In [3]:
#Split the Dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

#Encode labels
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

#70%-Training/10%-validation,20%-Testing
x = df['tweet_clean']
y = df['label_encoded']

#First split off test (20%)
X_temp, X_test, y_temp, y_test, idx_temp, idx_test = train_test_split(
    x, y, df.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

#Then split temp into train (87.5%) and val (12.5%)
X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X_temp, y_temp, idx_temp,
    test_size=0.125,
    random_state=42,
    stratify=y_temp
)

print(f"\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print("\nTrain label distribution:")
print(y_train.value_counts())
print("\nVal label distribution:")
print(y_val.value_counts())
print("\nTest label distribution:")
print(y_test.value_counts())


Label mapping: {'fact': np.int64(0), 'fake': np.int64(1), 'neutral': np.int64(2)}

Train: 4364 | Val: 624 | Test: 1248

Train label distribution:
label_encoded
2    1532
1    1416
0    1416
Name: count, dtype: int64

Val label distribution:
label_encoded
2    219
1    203
0    202
Name: count, dtype: int64

Test label distribution:
label_encoded
2    438
1    405
0    405
Name: count, dtype: int64


**Cell 4: Baseline model, TF-IDF + Logistic Regression.**

A baseline is a simple model that sets the bar of which the advanced model must surpass.

What this cell does:

1. TF-IDF vectorizer turns cleaned text into a vector of numbers.
2. The vectorizer is fitted on training data only, then used to transform validation and test in order to avoid leaking info from useen data.
3. Trains a Logistic Regression classifier.
4. Evaluates on the validation set and prints the macro-F1

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

#TF-IDF Baseline(Fit on train Only)
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

x_train_tfidf = tfidf.fit_transform(X_train)
x_val_tfidf = tfidf.transform(X_val)
x_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF vocab size: {len(tfidf.vocabulary_)}")
print(f"Train matrix: {x_train_tfidf.shape}")

#Train baseline
lr = LogisticRegression(max_iter=2000, random_state=42, n_jobs=-1)
lr.fit(x_train_tfidf, y_train)

#Validation F1
val_pred = lr.predict(x_val_tfidf)
val_f1 = f1_score(y_val, val_pred, average='macro')
print(f"\n[Baseline LR] Validation F1 (macro): {val_f1:.4f}")
print(classification_report(y_val, val_pred, target_names=le.classes_))




TF-IDF vocab size: 8000
Train matrix: (4364, 8000)

[Baseline LR] Validation F1 (macro): 0.5434
              precision    recall  f1-score   support

        fact       0.56      0.54      0.55       202
        fake       0.50      0.55      0.53       203
     neutral       0.57      0.53      0.55       219

    accuracy                           0.54       624
   macro avg       0.54      0.54      0.54       624
weighted avg       0.55      0.54      0.54       624



From the Output:

Baseline validation macro-F1 = 0.543, 54% accuracy

**Cell 5: Fine-tune AfriBERTa (main model)**

AfriBERTa (castorini/afriberta_large) is a transformer pre-trained on many African languages, including Swahili, which makes it a sensible choice for Kenyan text.

What this cell does:

1. Imports Pytorch and Hugging Face transformers.
2. Tokenizer: loads AfriBERTa's tokenizer and converts every tweet in train, validation and test into token IDs, truncated or padded to a fixed 128 tokens(MAX_LEN)
3. Misinfordataset class: wraps the tokenized data as a PyTorch Dataset, so each item returns the token tensors plus its label.
4. Model: loads AfriBERTa with a fresh 3-way classification head(num_labels=3)
5. compute_metrics: after each epoch, computes macro-F1 and accuracy.
6. trainer.train() runs the training

In [5]:
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)

#Tokenize (AfriBERTa vocab is pre-trained)
MODEL_NAME = "castorini/afriberta_large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 128

train_enc = tokenizer(list(X_train), truncation=True, padding='max_length', max_length=MAX_LEN)
val_enc = tokenizer(list(X_val), truncation=True, padding='max_length', max_length=MAX_LEN)
test_enc = tokenizer(list(X_test), truncation=True, padding='max_length', max_length=MAX_LEN)

#Wrapping the Dataset
class Misinfodataset(Dataset):
  def __init__(self, encodings, labels):
    self.encodings = encodings
    self.labels = labels.values

  def __getitem__(self, idx):
    item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
    item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
    return item
  def __len__(self):
    return len(self.labels)

train_dataset = Misinfodataset(train_enc, y_train)
val_dataset = Misinfodataset(val_enc, y_val)
test_dataset = Misinfodataset(test_enc, y_test)

#model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3
)

#Metrics
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = logits.argmax(-1)
  return{
      "f1_macro": f1_score(labels, preds, average='macro'),
      "accuracy": (preds == labels).mean()
  }

#Training
args = TrainingArguments(
    output_dir="./afriberta_misinfo",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=1e-5, #small, so the pre-trained knowledge is not overwritten
    #warmup_ratio=0.1,
    weight_decay=0.05, #Regularisation against overfitting
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()




config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 1.55MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  503MB            

... (6 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=819, training_loss=0.9079260843577402, metrics={'train_runtime': 122.0478, 'train_samples_per_second': 107.269, 'train_steps_per_second': 6.71, 'total_flos': 718637986759680.0, 'train_loss': 0.9079260843577402, 'epoch': 3.0})

**Cell 6: Final evaluation on the test set, overall and by language**

1. trainer.evaluate(test_dataset) scores the best model on the unseen test set and prints loss, macro-F1 and accuracy.
2. trainer.predict(...) produces the predicted class for each test tweet (argmax picks the class with the highest score), then prints a per-class report.
3. Per-Languaga breakdown: Looks up each test tweet's language using the saved idx_test, then for English, Swahili and Sheng seperately prints the macro-F1 and class report. This shows whether the model works equally well across languages.

In [6]:
#Test set:
test_results = trainer.evaluate(test_dataset)
print("Test results:", test_results)

#Per-class breakdown
preds = trainer.predict(test_dataset).predictions.argmax(-1)
print("\n=== Overall Test Report ===")
print(classification_report(y_test, preds, target_names=le.classes_))

#Per-language breakdown
test_lang = df.loc[idx_test, 'language'].values

for lang in ['English', 'Swahili', 'sheng']:
    mask = (test_lang == lang)
    if mask.sum() == 0:
        continue
    print(f"\n=== {lang} ({mask.sum()} samples) ===")
    print(f"F1 macro: {f1_score(y_test[mask], preds[mask], average='macro'):.4f}")
    print(classification_report(y_test[mask], preds[mask], target_names=le.classes_))


Training Loss,Validation Loss,Epoch,F1 Macro,Accuracy
0.833258,0.898970,3,0.589472,0.589744


Test results: {'eval_loss': 0.8989697694778442, 'eval_f1_macro': 0.5894723556096101, 'eval_accuracy': 0.5897435897435898}



=== Overall Test Report ===
              precision    recall  f1-score   support

        fact       0.61      0.62      0.62       405
        fake       0.56      0.55      0.55       405
     neutral       0.60      0.60      0.60       438

    accuracy                           0.59      1248
   macro avg       0.59      0.59      0.59      1248
weighted avg       0.59      0.59      0.59      1248


=== English (615 samples) ===
F1 macro: 0.5772
              precision    recall  f1-score   support

        fact       0.58      0.62      0.60       197
        fake       0.56      0.54      0.55       209
     neutral       0.58      0.58      0.58       209

    accuracy                           0.58       615
   macro avg       0.58      0.58      0.58       615
weighted avg       0.58      0.58      0.58       615


=== Swahili (48 samples) ===
F1 macro: 0.5359
              precision    recall  f1-score   support

        fact       0.75      0.60      0.67         5
     

From Output:
The model scores 0.94 F1 on neutral simply because almost every Swahili tweet is neutral. Predicting "neutral" most of the time would already score highly.
This indicates a labelling problem in the dataset.

English and Sheng performance is a reliable measure. Swahili perormance is unknown, because after de-duplication there was not enough Swahili data to train on or to test fairly.

**Cell 7: Check for near-duplicate tweets**
This cell matches Cell 2, only catches identical text.

what the cell does:
1. Prints the current row count and language counts
2. builds a TF-IDF representation of every cleaned tweet
3. Computes ***cosine similarity*** between every pair of tweets
4. Sets each tweet's similarity to itself to 0 so it doesn't match itself.
5. Flags a tweet as having a near-duplicate if any other tweet scores above 0.85
6. Counts the flagged tweets overall and per language.

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print(f"Before dedup: {len(df)} rows")
print(df['language'].value_counts())

# Vectorize all tweets
tfidf_check = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),  # bigrams help catch near-dupes
    min_df=1,
    max_df=0.99,
)
X_check = tfidf_check.fit_transform(df['tweet_clean'])

# Compute pairwise cosine similarity (only within same language to avoid cross-lingual noise)
# Note: for 9,000 rows, a 9000x9000 matrix is ~324MB — manageable
sim_matrix = cosine_similarity(X_check)

# Mask out self-similarities
np.fill_diagonal(sim_matrix, 0)

# For each row, is there ANY other row with similarity > 0.85?
near_dupe_threshold = 0.85
has_near_dupe = (sim_matrix > near_dupe_threshold).any(axis=1)

print(f"Rows with a near-duplicate (cosine > {near_dupe_threshold}): {has_near_dupe.sum()}")
print(f"Near-dupes by language:")
for lang in ['English', 'sheng', 'Swahili']:
    mask = df['language'] == lang
    print(f"  {lang}: {has_near_dupe[mask].sum()} of {mask.sum()} rows")

Before dedup: 6236 rows
language
sheng      2999
English    2999
Swahili     238
Name: count, dtype: int64
Rows with a near-duplicate (cosine > 0.85): 74
Near-dupes by language:
  English: 34 of 2999 rows
  sheng: 39 of 2999 rows
  Swahili: 1 of 238 rows
